In [2]:
import sys
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "data-pipeline" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import *
from src.features import *
from src.paths import (
    PROJECT_ROOT, MARKET_JSONL, PROCESSED_DIR, ORDERBOOK_DIR, FILTERED_TOKEN_IDS, ensure_dirs
)

ensure_dirs()
print("Using project root:", PROJECT_ROOT)

market_jsonl_path = MARKET_JSONL
markets_path = PROCESSED_DIR / "markets.parquet"
tokens_path = PROCESSED_DIR / "tokens.parquet"
filtered_tokens_path = FILTERED_TOKEN_IDS
raw_orderbook_dir = ORDERBOOK_DIR
feat_orderbook_dir = PROCESSED_DIR / "orderbook"

Using project root: C:\Users\Daron\coding\inefficiency-prediction


In [3]:
# TO CREATE MARKETS TABLE
markets_df = read_market_to_df(market_jsonl_path)

# Show the resulting table.
print(markets_df.shape)
markets_df.head()

(356736, 27)


,market_id,condition_id,question,slug,description,start_date,end_date,closed_time,active,closed,...,market_best_bid,market_best_ask,last_trade_price,resolution_source,uma_resolution_status,created_at,updated_at,n_outcomes,outcomes_raw,outcome_prices_raw
0,521502,0x0f8fa51925ce2d46f6767a268e98890258592b193766...,Utah vs. Capitals,nhl-utah-wsh-2025-02-09,"In the upcoming NHL game, scheduled for Februa...",2025-02-03T07:03:08.57174Z,2025-02-16T17:30:00Z,2025-02-09 22:57:34+00,True,True,...,0.999,1.000,1.0,https://www.nhl.com/,resolved,2025-02-03T07:00:38.152457Z,2025-02-10T21:13:10.419007Z,2,"[""Utah"", ""Capitals""]","[""1"", ""0""]"
1,521503,0x28010eca4446fdd1f3bae2b932fe0071d27dcbab290d...,Lightning vs. Canadiens,nhl-tb-mon-2025-02-09,"In the upcoming NHL game, scheduled for Februa...",2025-02-03T07:03:18.464733Z,2025-02-16T18:00:00Z,2025-02-09 23:17:30+00,True,True,...,0.999,1.000,1.0,https://www.nhl.com/,resolved,2025-02-03T07:00:52.596765Z,2025-02-10T21:43:12.615134Z,2,"[""Lightning"", ""Canadiens""]","[""1"", ""0""]"
2,521485,0xee068b9b7d89d0a68b1b1940a143e961c5752e294fa9...,Will Elon tweet less than 500 times Jan 31 - F...,will-elon-tweet-less-than-500-times-jan-31-feb-7,This market will resolve according to the numb...,2025-02-03T15:43:11.110902Z,2025-02-07T12:00:00Z,2025-02-05 21:17:55+00,True,True,...,NaN,0.001,1.0,,resolved,2025-02-03T03:32:37.545769Z,2025-02-06T21:05:02.520187Z,2,"[""Yes"", ""No""]","[""0"", ""1""]"
3,521488,0x02c7e0242f53ac51f16c0597d4db9709e1b3bc507faa...,Will Elon tweet 500-524 times Jan 31 - Feb 7?,will-elon-tweet-500-524-times-jan-31-feb-7,This market will resolve according to the numb...,2025-02-03T15:43:35.984934Z,2025-02-07T12:00:00Z,2025-02-06 01:18:23+00,True,True,...,NaN,0.001,1.0,,resolved,2025-02-03T03:32:38.691185Z,2025-02-07T00:09:03.313579Z,2,"[""Yes"", ""No""]","[""0"", ""1""]"
4,521489,0x1b2768ac62dfea779405822eb1fe67691eac684cfba5...,Will Elon tweet 525-549 times Jan 31 - Feb 7?,will-elon-tweet-525-549-times-jan-31-feb-7,This market will resolve according to the numb...,2025-02-03T15:44:00.85876Z,2025-02-07T12:00:00Z,2025-02-06 17:28:37+00,True,True,...,NaN,0.001,1.0,,resolved,2025-02-03T03:32:39.116039Z,2025-02-07T15:31:11.091833Z,2,"[""Yes"", ""No""]","[""0"", ""1""]"


In [4]:
# TO SAVE MARKETS TABLE AS "markets.parquet"
markets_df.to_parquet(markets_path, index=False)

In [5]:
# TO READ MARKETS TABLE FROM "markets.parquet"
markets_df = pd.read_parquet(markets_path)
print(markets_df.shape)
print(markets_df.columns)

(356736, 27)
Index(['market_id', 'condition_id', 'question', 'slug', 'description',
       'start_date', 'end_date', 'closed_time', 'active', 'closed', 'archived',
       'accepting_orders', 'enable_order_book', 'neg_risk', 'fee_decimal',
       'volume', 'volume_clob', 'market_best_bid', 'market_best_ask',
       'last_trade_price', 'resolution_source', 'uma_resolution_status',
       'created_at', 'updated_at', 'n_outcomes', 'outcomes_raw',
       'outcome_prices_raw'],
      dtype='str')


In [6]:
# TO FILTER MARKETS TABLE BY END DATE AND VOLUME, AND GET TOP 70 MARKETS
markets_df = markets_df.dropna(subset=["question", "market_id", "end_date", "closed_time"])
markets_filtered_df = markets_df[markets_df["end_date"] > "2026-01-01"].sort_values("volume", ascending=False).head(70)
relevant_market_ids = set(markets_filtered_df["market_id"].astype(str))
len(relevant_market_ids)

70

In [7]:
# TO CREATE TOKENS TABLE
tokens_df = read_token_to_df(market_jsonl_path)

print(tokens_df.shape)
tokens_df.head()

(713472, 20)


,market_id,token_id,outcome_index,outcome,outcome_price_initial,question,slug,description,start_date,end_date,closed_time,active,closed,archived,accepting_orders,enable_order_book,neg_risk,fee_decimal,volume,volume_clob
0,521502,4278735810733499450033805996998150780100263473...,0,Utah,1.0,Utah vs. Capitals,nhl-utah-wsh-2025-02-09,"In the upcoming NHL game, scheduled for Februa...",2025-02-03T07:03:08.57174Z,2025-02-16T17:30:00Z,2025-02-09 22:57:34+00,True,True,False,False,True,False,0.02,94610.905573,94610.905573
1,521502,1111661147715280529758155544116704456172196124...,1,Capitals,0.0,Utah vs. Capitals,nhl-utah-wsh-2025-02-09,"In the upcoming NHL game, scheduled for Februa...",2025-02-03T07:03:08.57174Z,2025-02-16T17:30:00Z,2025-02-09 22:57:34+00,True,True,False,False,True,False,0.02,94610.905573,94610.905573
2,521503,5167092641191119841414124845941123936979062682...,0,Lightning,1.0,Lightning vs. Canadiens,nhl-tb-mon-2025-02-09,"In the upcoming NHL game, scheduled for Februa...",2025-02-03T07:03:18.464733Z,2025-02-16T18:00:00Z,2025-02-09 23:17:30+00,True,True,False,False,True,False,0.02,134897.501228,134897.501228
3,521503,8574320122684556697660599417444078755289261476...,1,Canadiens,0.0,Lightning vs. Canadiens,nhl-tb-mon-2025-02-09,"In the upcoming NHL game, scheduled for Februa...",2025-02-03T07:03:18.464733Z,2025-02-16T18:00:00Z,2025-02-09 23:17:30+00,True,True,False,False,True,False,0.02,134897.501228,134897.501228
4,521485,1117088797318410927651791754301246752693394101...,0,Yes,0.0,Will Elon tweet less than 500 times Jan 31 - F...,will-elon-tweet-less-than-500-times-jan-31-feb-7,This market will resolve according to the numb...,2025-02-03T15:43:11.110902Z,2025-02-07T12:00:00Z,2025-02-05 21:17:55+00,True,True,False,False,True,True,NaN,190849.597987,190849.597987


In [8]:
# TO SAVE TOKENS TABLE AS "tokens.parquet"
tokens_df.to_parquet(tokens_path, index=False)

In [14]:
# TO READ TOKENS TABLE
tokens_df = pd.read_parquet(tokens_path)

print(tokens_df.shape)
print(tokens_df.columns)

(713472, 20)
Index(['market_id', 'token_id', 'outcome_index', 'outcome',
       'outcome_price_initial', 'question', 'slug', 'description',
       'start_date', 'end_date', 'closed_time', 'active', 'closed', 'archived',
       'accepting_orders', 'enable_order_book', 'neg_risk', 'fee_decimal',
       'volume', 'volume_clob'],
      dtype='str')


In [15]:
# TO FILTER TOKENS TABLE TO ONLY INCLUDE TOKENS FROM THE FILTERED MARKETS
tokens_filtered_df = tokens_df[
    tokens_df["market_id"].astype(str).isin(relevant_market_ids)
].copy()
tokens_filtered_df.shape

(140, 20)

In [12]:
# TO SAVE THE FILTERED TOKEN IDs TO A PARQUET FILE
relevant_token_ids = set(tokens_filtered_df["token_id"].astype(str))

pd.DataFrame({"token_id": sorted(relevant_token_ids)}).to_parquet(filtered_tokens_path, index=False)

In [13]:
await extract_orderbook_from_ids_async(
    token_ids=relevant_token_ids,
    start_date="2025-10-14",
    end_date="2026-03-31",
    output_path=raw_orderbook_dir,
    max_concurrent=8
)

Processing token_id=27110946212212220543885771558729072300747406628273094439631175059267259915499, 1/140
Processing token_id=63864522346594368809423416425052844778517942296857413266450057529414766826445, 2/140
Processing token_id=14705029091296171110674089399091656949154521086160164017708184564241023149952, 3/140
Processing token_id=57969484623906654162647807925252512006821410423354182075797975676232320775740, 4/140
Processing token_id=113930583156121915694796889379494567927783442690238586410669568657607120812138, 5/140
Processing token_id=15002165665234016876486282630507392757515779060349352387653810600083538604826, 6/140
Processing token_id=108381671107274799941073921577360862012191130720671718707997750576881298527847, 7/140
Processing token_id=18491832759883472528052920586615186501076720389035092804420992035191801268891, 8/140
Saved 9,065 rows for token_id=18491832759883472528052920586615186501076720389035092804420992035191801268891 to C:\Users\Daron\coding\inefficiency-prediction\d